# Мощность статистического критерия

На основе [Stat power.ipynb — Академия аналитиков Авито](https://github.com/avito-tech/applied_statistics/blob/main/Stat%20power.ipynb).

В этой задаче **одна выборка**: проверяем H₀: p = p₀ против H₁: p > p₀. Это не сравнение двух независимых групп A/B-теста.

**Как пользоваться:** поменяй значения только в следующей ячейке и запусти **Run All**. Графики появятся ниже, если `SHOW_PLOTS = True`.


In [ ]:
from scipy.stats import binom
import numpy as numpy
from matplotlib import pyplot


## 1. ВВОД ДАННЫХ


In [ ]:
N = 30               # количество испытаний / наблюдений
MU_H0 = 0.50         # вероятность успеха при H0 (50%)
MU_ALT = 0.60        # ожидаемая вероятность успеха при H1 (60%)
ALPHA = 0.05         # уровень значимости
MIN_POWER = 0.80     # целевая мощность для графиков
SHOW_PLOTS = True    # False — только числа


## 2. Расчёт мощности

Мощность — вероятность отвергнуть H₀, когда верна альтернатива. В коде ниже сохранена исходная функция из лекции.


In [ ]:
def get_stat_power(N, mu_h0, mu_alternative, alpha):
    '''Вычисляет статистическую мощность критерия для биномиального распределения
    
    Параметры:
        N - количество бернуллиевских экспериментов (размер выборки)
        mu_h0 - вероятность успеха в нулевой гипотезе
        mu_alternative - предполагаемая вероятность успеха в эксперименте
        alpha - уровень значимости критерия
    '''
    binom_h0 = binom(n=N, p=mu_h0)
    binom_alternative = binom(n=N, p=mu_alternative)

    # вычисляем критическое значение
    critical_value = binom_h0.ppf(1 - alpha) + 1
    # вычисляем мощность по формуле
    return 1 - binom_alternative.cdf(critical_value - 1) 

In [ ]:
critical_value = binom(n=N, p=MU_H0).ppf(1 - ALPHA) + 1
fpr = 1 - binom(n=N, p=MU_H0).cdf(critical_value - 1)
power = get_stat_power(N, MU_H0, MU_ALT, alpha=ALPHA)

print(f"Критическая область: Q >= {critical_value:.0f}")
print(f"Ошибка I рода (фактическая): {fpr:.2%}")
print(f"Мощность: {power:.2%}")


## 3. График: H₀, H₁ и критическая область

Как в лекции, сравниваем два биномиальных распределения и выделяем область отвержения H₀.


In [ ]:
if SHOW_PLOTS:
    x_grid = numpy.arange(0, N + 1)
    crit_reg = x_grid >= critical_value
    binom_h0 = binom(n=N, p=MU_H0)
    binom_alternative = binom(n=N, p=MU_ALT)

    pyplot.figure(figsize=(10, 5))
    # строим распределение при H0 и критическую область
    probs_h0 = binom_h0.pmf(x_grid)
    pyplot.vlines(x_grid, 0, probs_h0, linewidth=4, label="H0", alpha=0.65)

    # строим распределение при mu > 0.5
    probs_alternative = binom_alternative.pmf(x_grid)
    pyplot.vlines(x_grid + 0.15, 0, probs_alternative, linewidth=4, label="H1", alpha=0.65)
    pyplot.axvspan(critical_value - 0.5, N + 0.5, alpha=0.1, label="Критическая область")
    pyplot.xlabel("Число успехов")
    pyplot.ylabel("Вероятность")
    pyplot.title("Распределения при H0 и H1")
    pyplot.legend()
    pyplot.show()


## 4. График: как мощность зависит от размера выборки


In [ ]:
if SHOW_PLOTS:
    n_grid = numpy.arange(10, max(600, N * 10) + 1, 10)
    power_grid = get_stat_power(n_grid, MU_H0, MU_ALT, alpha=ALPHA)

    pyplot.figure(figsize=(10, 5))
    pyplot.plot(n_grid, power_grid, label="Мощность")
    pyplot.axhline(MIN_POWER, ls="--", label=f"Цель: {MIN_POWER:.0%}")
    # считаем минимальный размер выборки для мощности в 80%
    fitting_n = n_grid[power_grid >= MIN_POWER]
    if fitting_n.size:
        pyplot.axvline(fitting_n.min(), ls=":", label=f"Размер выборки ≈ {fitting_n.min()}")
    pyplot.xlabel("Число наблюдений")
    pyplot.ylabel("Мощность")
    pyplot.ylim(0, 1.02)
    pyplot.title("Мощность и размер выборки")
    pyplot.legend()
    pyplot.show()
